# Test Map Integration with Find User Feature

This notebook tests the complete map integration flow using API calls only.

**Features tested:**
- Check if map is uploaded and active
- Verify cameras have map positions
- Simulate user detection at camera
- Search user and verify map_position data is returned

**Before running:**
1. Make sure backend is running
2. Upload a map via Cameras → Map tab
3. Position cameras on the map
4. Login as ORG ADMIN

## Setup

In [6]:
import requests
import os
from datetime import datetime, timezone
from dotenv import load_dotenv

load_dotenv()

BACKEND_URL = os.getenv('SO_BACKEND_API_URL', 'http://localhost:7091')
session = requests.Session()
session.headers.update({"accept": "application/json"})

print('Setup complete!')
print(f'Backend API: {BACKEND_URL}')

Setup complete!
Backend API: http://localhost:7091


## Login

In [7]:
def login_to_backend(email=None, password=None, client_slug='humblebee'):
    if email is None:
        email = os.getenv('SO_ADMIN_EMAIL', 'admin@humblebee.ai')
    if password is None:
        password = os.getenv('SO_ADMIN_PASSWORD', 'admin123')

    print(f'Logging in as {email} to org "{client_slug}"...')

    response = session.post(
        f'{BACKEND_URL}/api/auth/login',
        json={'email': email, 'password': password, 'client_slug': client_slug}
    )

    if response.status_code == 200:
        data = response.json()
        token = data.get('token') or data.get('accessToken')
        session.headers.update({'Authorization': f'Bearer {token}'})
        print('Login successful')
        return token, client_slug
    else:
        print(f'Login failed: {response.status_code}')
        return None, None

auth_token, slug = login_to_backend(
    email='humblebee@gmail.com',
    password='Humblebee2025@',
    client_slug='humblebee'
)

Logging in as humblebee@gmail.com to org "humblebee"...
Login successful


## Get Active Map

In [ ]:
def get_active_map(slug):
    r = session.get(f"{BACKEND_URL}/api/org/{slug}/maps/active")
    if r.status_code == 200:
        return r.json().get('data')
    elif r.status_code == 404:
        return None
    else:
        r.raise_for_status()

active_map = get_active_map(slug)
if active_map:
    print(f"Active Map Found:")
    print(f"  ID: {active_map.get('id')}")
    print(f"  Name: {active_map.get('map_name')}")
    print(f"  Dimensions: {active_map.get('image_width')}x{active_map.get('image_height')}")
    print(f"  Cameras positioned: {len(active_map.get('camera_positions', []))}")

    if active_map.get('camera_positions'):
        print("\n  Cameras on map:")
        for pos in active_map['camera_positions']:
            print(f"    - {pos['camera_name']} at ({pos['x_coordinate']}, {pos['y_coordinate']})")
else:
    print("No active map found")
    print("Please upload a map using: Cameras → Map tab")

Active Map Found:
  ID: 1
  Name: asosiy
  Dimensions: 800x600
  Cameras positioned: 2

  Cameras on map:
    - new one at (699, 173)
    - new two at (178, 373)


## Get Cameras

In [9]:
def list_org_cameras(slug):
    r = session.get(f"{BACKEND_URL}/api/org/{slug}/cameras")
    r.raise_for_status()
    return r.json()

cameras = list_org_cameras(slug)
print(f"Found {len(cameras)} cameras")
for cam in cameras[:5]:
    print(f"  ID: {cam.get('id')}, Name: {cam.get('name')}, Location: {cam.get('location')}")

# Pick first camera for testing
test_camera = cameras[0] if cameras else None
if test_camera:
    print(f"\nUsing test camera: {test_camera['name']}")

Found 2 cameras
  ID: 14, Name: new one, Location: qabul
  ID: 25, Name: new two, Location: somewhere

Using test camera: new one


## Get Users

In [10]:
def list_org_users(slug, page=1, limit=50):
    r = session.get(f"{BACKEND_URL}/api/org/{slug}/users", params={"page": page, "limit": limit})
    r.raise_for_status()
    return r.json()

users = list_org_users(slug)
print(f"Found {len(users)} users")
for u in users[:5]:
    print(f"  ID: {u.get('id')}, Name: {u.get('full_name')}")

# Pick first user for testing
test_user = users[0] if users else None
if test_user:
    print(f"\nUsing test user: {test_user['full_name']}")

Found 41 users
  ID: 81, Name: Person B
  ID: 80, Name: Integration Test User 08:47:23
  ID: 79, Name: Integration Test User 08:45:30
  ID: 78, Name: Integration Test User 08:30:15
  ID: 77, Name: Integration Test User 08:28:52

Using test user: Person B


## Helper Function

In [11]:
def iso_now():
    return datetime.now(timezone.utc).isoformat().replace('+00:00', 'Z')

print('Helper functions ready!')

Helper functions ready!


## Test 1: Simulate User Detection at Camera

This simulates what the face recognition system does when it detects a user.

In [12]:
if test_user and test_camera:
    detection_payload = {
        "user_name": test_user['full_name'],
        "camera_name": test_camera['name'],
        "timestamp": iso_now(),
        "status": "in"
    }

    print(f"Simulating user detection:")
    print(f"  User: {detection_payload['user_name']}")
    print(f"  Camera: {detection_payload['camera_name']}")
    print(f"  Time: {detection_payload['timestamp']}")

    r = session.post(
        f"{BACKEND_URL}/api/org/{slug}/user-locations",
        json=detection_payload
    )

    if r.status_code == 201:
        result = r.json()
        print('\nUser detection recorded successfully!')
        print(f"  Location ID: {result.get('id')}")
    else:
        print(f'\nError: {r.status_code}')
        print(r.text[:300])
else:
    print('Missing test data - skipping detection')

Simulating user detection:
  User: Person B
  Camera: new one
  Time: 2025-11-27T15:33:48.939420Z

User detection recorded successfully!
  Location ID: None


## Test 2: Search for User (KEY TEST)

This tests what the Find User page does - verify that map_position data is returned.

In [13]:
if test_user:
    search_term = test_user['full_name'].split()[0]
    print(f"Searching for: {search_term}...\n")

    r = session.get(
        f"{BACKEND_URL}/api/org/{slug}/user-locations/search",
        params={"query": search_term}
    )

    if r.status_code == 200:
        data = r.json()
        print(f"Search API returned successfully\n")
        print("=" * 70)

        if data.get('data'):
            for result in data['data']:
                user = result.get('user', {})
                location = result.get('location')
                is_recent = result.get('is_recent', False)

                print(f"\nUser: {user.get('full_name')} (ID: {user.get('id')})")
                print(f"  Recent: {'Yes' if is_recent else 'No (stale)'}")

                if location:
                    print(f"  Camera: {location.get('camera_name')}")
                    print(f"  Location: {location.get('camera_location', 'N/A')}")
                    print(f"  Detected: {location.get('detected_at')}")

                    # KEY CHECK: map position data
                    map_pos = location.get('map_position')
                    if map_pos:
                        print(f"\n  MAP POSITION FOUND:")
                        print(f"    Map ID: {map_pos.get('map_id')}")
                        print(f"    X: {map_pos.get('x')}")
                        print(f"    Y: {map_pos.get('y')}")
                        print(f"\n  SUCCESS! Map integration is working!")
                        print(f"  The frontend will show the camera icon on the map.")
                    else:
                        print(f"\n  No map position in response")
                        print(f"  Camera not positioned on map yet.")
                else:
                    print(f"  No location data available")

            print("\n" + "=" * 70)
        else:
            print("No results found")
    else:
        print(f'Error: {r.status_code}')
        print(r.text[:300])
else:
    print('No test user - skipping search')

Searching for: Person...

Search API returned successfully


User: Person B (ID: 81)
  Recent: Yes
  Camera: new one
  Location: qabul
  Detected: 2025-11-27T15:33:48.939Z

  MAP POSITION FOUND:
    Map ID: 1
    X: 699
    Y: 173

  SUCCESS! Map integration is working!
  The frontend will show the camera icon on the map.



## Summary

### What Just Happened:

If you see "SUCCESS! Map integration is working!" above:

1. Backend correctly joins camera positions with user locations
2. API returns `map_position` data in the response
3. Frontend will display camera icon on map when users search

### Production Flow:

```
Face Recognition → Detects User → POST /api/org/{slug}/user-locations
                                      ↓
User searches   → Find User Page  → GET /api/org/{slug}/user-locations/search
                                      ↓
                   Frontend Shows: Camera feed + Map + Camera icon position
```

### Troubleshooting:

- **No map found**: Upload map via Cameras → Map tab
- **No camera positions**: Position cameras via Cameras → Map tab → Position Cameras
- **Old detection**: Make a new detection (within last 30 minutes)
- **Login failed**: Update credentials in Login cell